# 01 — Data Initialization

Audit BDD100K sources and freeze the four disjoint data roles before preprocessing or training. Convert native labels and prepare verified Raw development references while keeping final-test pixels unopened.

- **Inputs:** Original BDD detection images/annotations and `notebooks/test_exposure.json`.
- **Outputs:** `runs/data-v1/`: immutable protocol, converted development labels and Raw references; `docs/data-v1/`: compact audits.
- **Prerequisites:** Project `.venv` kernel, installed study dependencies, local BDD sources and writable new output roots.

**Outline**

1. [Configuration and Preflight](#configuration-preflight)
2. [Source and Role Audit](#source-role-audit)
3. [Distributions and Leakage Diagnostics](#distributions-leakage)
4. [Label Conversion and Raw References](#label-conversion)
5. [Export and Integrity Summary](#export-integrity)

Set paths in section 1, then **Restart & Run All**. Only this phase executes.
Missing or mismatched required artifacts fail; upstream phases are never launched.
The historical six-class taxonomy, including railway `train`, stays unchanged.

<a id="configuration-preflight"></a>

## 1. Configuration and Preflight

Configure source/output paths and allocation settings. Existing differing manifests require a new version/root.

In [ ]:
import gzip
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from IPython.utils.capture import capture_output

from vehicle_detection.common import file_hash, read_json, require_paths, write_json
from vehicle_detection.data import CLASSES, convert_labels, scan_bdd
from vehicle_detection.offline import export_variant, verify_alignment
from vehicle_detection.protocol import (
    FRACTIONS,
    build_protocol,
    class_condition_audit,
    development_records,
    freeze_protocol,
    validate_protocol,
)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
DATA_ROOT = ROOT / "data/bdd100k"
RUN_ROOT = ROOT / "runs/data-v1"
REPORT_ROOT = ROOT / "docs/data-v1"
SEED, ALLOCATION_ATTEMPTS = 2026, 64
require_paths(data=DATA_ROOT, exposures=ROOT / "notebooks/test_exposure.json")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 90, "font.size": 9})
print("WSL-native dataset:", DATA_ROOT.resolve())

<a id="source-role-audit"></a>

## 2. Source and Role Audit

Discovery inventories paths and annotation schemas. Official-val checks use annotations/file existence only; final-test images are never decoded, hashed or previewed.

In [ ]:
records, discovery, sources = scan_bdd(DATA_ROOT, strict=False)
exposure_path = ROOT / "notebooks/test_exposure.json"
exposures = read_json(exposure_path)
excluded_ids = sorted({sample_id for ids in exposures.values() for sample_id in ids})
protocol = build_protocol(
    records,
    sources,
    DATA_ROOT,
    excluded_ids,
    seed=SEED,
    attempts=ALLOCATION_ATTEMPTS,
    discovery=discovery,
)
freeze_protocol(RUN_ROOT / "protocol.json", protocol)
assert validate_protocol(read_json(RUN_ROOT / "protocol.json")) == protocol
print("Manifest identity:", protocol["manifest_id"])
display(pd.Series(discovery, name="source discovery"))
display(pd.Series(protocol["label_audit"], name="box audit"))
display(pd.DataFrame(protocol["exclusions"], columns=["id", "reason"]).groupby("reason").size())
print("Prior exposed IDs:", len(excluded_ids))
display(protocol["group_audit"])

<a id="distributions-leakage"></a>

## 3. Distributions and Leakage Diagnostics

Describe frozen roles and disclose drift/support without rebalancing them. Official-train evidence alone determines allocation.

### 3.1 Class Presence and Instances

In [ ]:
counts = pd.DataFrame(protocol["summaries"]).fillna(0).astype(int)
roles = list(FRACTIONS) + ["final_test"]
display(counts.loc[["images", "negative", *[f"presence:{c}" for c in CLASSES]]].T)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, prefix, title in zip(
    axes,
    ["presence", "instances"],
    ["Class image presence / image", "Vehicle instances / image"],
    strict=True,
):
    data = counts.loc[[f"{prefix}:{c}" for c in CLASSES], roles].div(counts.loc["images", roles])
    data.index = CLASSES
    data.plot.bar(ax=ax, title=title, rot=30)
fig.tight_layout()
fig.savefig(REPORT_ROOT / "class_distributions.png")
plt.show()

### 3.2 Conditions, Sizes and Negative Frames

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, key in zip(axes, ["weather", "timeofday", "scene"], strict=True):
    data = counts.loc[
        [i for i in counts.index if i.startswith(key + ":")],
        ["official_train", "official_val", "final_test"],
    ]
    data.index = data.index.str.split(":", n=1).str[1]
    data.div(counts.loc["images", data.columns]).plot.bar(ax=ax, title=key, rot=65)
fig.tight_layout()
fig.savefig(REPORT_ROOT / "condition_distributions.png")
plt.show()
display(counts.loc[[i for i in counts.index if i.startswith("missing_metadata:")]].T)
print("Official val/final test are descriptive annotation audits, not optimization targets.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
size_keys = [f"size_instances:{s}" for s in ["small", "medium", "large"]]
counts.reindex(size_keys, fill_value=0)[roles].div(counts.loc["images", roles]).plot.bar(
    ax=axes[0],
    title="Object size instances / image (native annotation pixels)",
    rot=20,
)
counts.loc[["negative"], roles].div(counts.loc["images", roles]).T.plot.bar(
    ax=axes[1],
    title="Negative image fraction",
    legend=False,
    rot=20,
)
fig.tight_layout()
fig.savefig(REPORT_ROOT / "size_and_background.png")
plt.show()
diagnostics = pd.DataFrame(protocol["diagnostics"])
display(diagnostics.loc[diagnostics.inadequate_support].sort_values(["feature", "role"]))
display(diagnostics.assign(abs_drift=diagnostics.delta_per_image.abs()).nlargest(12, "abs_drift"))
print("Support <5: report parent marginal; no separate subgroup inference. No oversampling.")
print("\n".join(protocol["limitations"]))

### 3.3 Class × Condition Support

Rare/zero support is descriptive. Cells below five class images do not support separate subgroup inference.

In [ ]:
joint = pd.DataFrame(class_condition_audit(protocol)).sort_values(
    ["role", "axis", "condition", "class"]
)
joint.to_csv(REPORT_ROOT / "class_condition_support.csv", index=False)
for axis in ["weather", "timeofday"]:
    data = joint[joint.axis == axis]
    conditions = sorted(data.condition.unique())
    fig, axes = plt.subplots(1, 4, figsize=(18, 4), layout="constrained")
    limit = max(1, data.conditional_presence_drift_pp.abs().max())
    for ax, role in zip(axes, roles, strict=True):
        selected = data[data.role == role]
        drift = selected.pivot(
            index="class", columns="condition", values="conditional_presence_drift_pp"
        ).reindex(index=CLASSES, columns=conditions)
        image_counts = selected.pivot(
            index="class", columns="condition", values="class_images"
        ).reindex(index=CLASSES, columns=conditions)
        box_counts = selected.pivot(index="class", columns="condition", values="instances").reindex(
            index=CLASSES, columns=conditions
        )
        heat = ax.imshow(drift, cmap="coolwarm", vmin=-limit, vmax=limit, aspect="auto")
        sample_sizes = selected.groupby("condition").condition_images.first()
        ax.set_xticks(
            range(len(conditions)), [f"{c}\nn={sample_sizes[c]}" for c in conditions], rotation=65
        )
        ax.set_yticks(range(len(CLASSES)), CLASSES)
        ax.set_title(role.replace("_", " "))
        for i in range(len(CLASSES)):
            for j in range(len(conditions)):
                ax.text(
                    j,
                    i,
                    f"{image_counts.iloc[i, j]}\n{box_counts.iloc[i, j]}",
                    ha="center",
                    va="center",
                    fontsize=6,
                )
    fig.colorbar(heat, ax=axes, label="Within-condition class drift (pp)")
    fig.suptitle(
        f"Class × {axis}: each cell = image presence / instances; final test metadata only"
    )
    fig.savefig(REPORT_ROOT / f"class_{axis}_support.png")
    plt.show()
display(joint[joint.rare_support].sort_values(["axis", "condition", "class", "role"]))
print("CSV includes every class/condition/role, including zero support, denominators and drift.")
print("Reference is all official train. These post-allocation audits do not alter frozen IDs.")

Read these plots as integrity/support diagnostics. They do not select models or justify Adaptive policies; final-test metadata never enters tuning.

<a id="label-conversion"></a>

## 4. Label Conversion and Raw References

Use the pinned BDD/Scalabel/Ultralytics converter with continuous-coordinate roundtrip checks. Raw is unchanged pixels followed by the shared JPEG95 encoding; only development inputs are exported.

In [ ]:
partitions = development_records(protocol, records, DATA_ROOT)
converted = RUN_ROOT / "conversion"
with capture_output() as conversion_log:
    conversion = convert_labels(partitions, converted, convention="continuous")
(RUN_ROOT / "conversion.log").write_text(conversion_log.stdout + conversion_log.stderr)
metadata = export_variant(
    partitions,
    converted,
    RUN_ROOT / "datasets/raw",
    "raw_reencoded",
    {"raw": {}},
    output_format="jpg",
    jpeg_quality=95,
)
assert verify_alignment([RUN_ROOT / "datasets/raw"]) == len(partitions["tune"])
assert not list((RUN_ROOT / "datasets/raw/images/test").iterdir())
print("Deep-verified development Raw images:", len(metadata))
display(conversion)
display(metadata[["preprocess_ms", "preparation_ms"]].agg(["mean", "median", "max"]))

<a id="export-integrity"></a>

## 5. Export and Integrity Summary

Publish compact membership/provenance alongside the immutable manifest. Verify split sizes, label/geometry checks and the absence of final-test exports.

In [ ]:
membership = pd.DataFrame(
    [
        {"role": role, **{k: row[k] for k in ["id", "group", "source_group"]}}
        for role, rows in protocol["splits"].items()
        for row in rows
    ]
).sort_values(["role", "id"])
member_path = REPORT_ROOT / "split_membership.csv.gz"
compressed = gzip.compress(membership.to_csv(index=False).encode(), mtime=0)
if member_path.exists() and member_path.read_bytes() != compressed:
    raise ValueError("Committed membership changed; version the protocol and review the diff")
member_path.write_bytes(compressed)
summary = {k: v for k, v in protocol.items() if k != "splits"}
summary.update(
    {
        "split_sizes": {role: len(rows) for role, rows in protocol["splits"].items()},
        "membership_sha256": file_hash(member_path),
        "exposure_ledger_sha256": file_hash(exposure_path),
        "local_full_manifest": "runs/data-v1/protocol.json",
        "class_condition_audit_sha256": file_hash(REPORT_ROOT / "class_condition_support.csv"),
        "raw_export": read_json(RUN_ROOT / "datasets/raw/export.json"),
        "raw_files_identity": read_json(RUN_ROOT / "datasets/raw/verified.json")["files_identity"],
    }
)
write_json(REPORT_ROOT / "protocol_summary.json", summary)
print("PASS: frozen roles, disclosed exclusions, official conversion, development-only Raw export")
display(pd.Series(summary["split_sizes"], name="frozen image counts"))

A successful summary establishes artifact dependencies for later notebooks. It does not run preprocessing research, model training or inference.